In [2]:
#20%
import numpy as np

In [16]:
def householder_qr_simple(A_init, b_init, eps=1e-12):
    A = A_init.copy().astype(float)
    b = b_init.copy().astype(float)
    n = len(A)

    Q = np.eye(n)
    # Q = I

    for r in range(n - 1):
      
        sigma = 0.0
        for i in range(r, n):
            sigma += A[i, r] ** 2

        if np.sqrt(sigma) <= eps:
            print(f"Matrice singulară")
            break
      
        if A[r, r] >= 0:
            sign_ar = 1.0 
        else: 
            sign_ar = -1.0

        k = -sign_ar * np.sqrt(sigma)

        beta = sigma - k * A[r, r]
        
        if abs(beta) < eps:
            continue

        #u=0 i<r 
        #u=arr-k i=r 
        #u=air i>r
        u = np.zeros(n)
        u[r] = A[r, r] - k
        for i in range(r + 1, n):
            u[i] = A[i, r]

        # A = Pr * A
        # transformarea coloanelor j=r+1,...n
        for j in range(r, n):
            gamma = 0.0
            for i in range(r, n):
                gamma += u[i] * A[i, j]
            gamma /= beta
            for i in range(r, n):
                A[i, j] -= gamma * u[i]

        # transformarea coloanei j=r
        A[r, r] = k
        for i in range(r + 1, n):
            A[i, r] = 0.0

        # b = Pr * b
        gamma_b = 0.0
        for i in range(r, n):
            gamma_b += u[i] * b[i]
        gamma_b /= beta
        for i in range(r, n):
            b[i] -= gamma_b * u[i]

        # Q = Pr * Q
        for j in range(n):
            gamma_Q = 0.0
            for i in range(r, n):
                gamma_Q += u[i] * Q[i, j]
            gamma_Q /= beta
            # qᵢⱼ = qᵢⱼ - γ·uᵢ
            for i in range(r, n):
                Q[i, j] -= gamma_Q * u[i]

      
    return Q.T, A, b


In [17]:
def back_substitution(R, y, eps=1e-12):
    n = len(y)
    x = np.zeros(n)
    for i in reversed(range(n)):
        s = 0.0
        for j in range(i + 1, n):
            s += R[i, j] * x[j]

        if abs(R[i, i]) < eps:
            raise ValueError("Matrice singulară")
        
        x[i] = (y[i] - s) / R[i, i]
    return x

In [18]:
def inverse_householder(A, Q, R, eps=1e-12):
    n = len(A)
    
    for i in range(n):
        if abs(R[i, i]) < eps:
            raise ValueError(f"Matricea A este singulară")
    
    A_inv = np.zeros((n, n))
    
    for j in range(n):
        e = np.zeros(n)
        e[j] = 1.0
        b = Q.T @ e
        
        x = back_substitution(R, b, eps)
        
        for i in range(n):
            A_inv[i, j] = x[i]
    
    return A_inv

In [19]:

np.set_printoptions(precision=6, suppress=True)
  
n = int(input("n: ") or 4)
eps = 1e-8
np.random.seed(42)  

A = np.random.rand(n, n)
s = np.random.rand(n)
b = np.zeros(n)
for i in range(n):  
    for j in range(n):
        b[i] += A[i, j] * s[j]

print("\nMatricea A:\n", A)
print("s:", s)
print("b = A*s:", b)

   
Q, R, b_trans = householder_qr_simple(A, b, eps)
x_H = back_substitution(R, Q.T @ b)
Q_lib, R_lib = np.linalg.qr(A)
x_QR = np.linalg.solve(R_lib, Q_lib.T @ b)

err_x_diff = np.linalg.norm(x_QR - x_H, 2)
err_Axb_H = np.linalg.norm(A @ x_H - b, 2)
err_Axb_QR = np.linalg.norm(A @ x_QR - b, 2)
err_xs_H = np.linalg.norm(x_H - s, 2) / np.linalg.norm(s, 2)
err_xs_QR = np.linalg.norm(x_QR - s, 2) / np.linalg.norm(s, 2)


A_inv_H = inverse_householder(A, Q, R)
A_inv_lib = np.linalg.inv(A)
err_inv = np.linalg.norm(A_inv_H - A_inv_lib, 2)


print("\n--- Results ---")
print("x_Householder =", x_H)
print("x_QR (library) =", x_QR)
print("||x_QR - x_H||2 =", err_x_diff)
print("\nErrors:")
print("||A x_H - b||2 =", err_Axb_H)
print("||A x_QR - b||2 =", err_Axb_QR)
print("||x_H - s|| / ||s|| =", err_xs_H)
print("||x_QR - s|| / ||s|| =", err_xs_QR)
print("\nInverse comparison:")
print("||A^-1_H - A^-1_lib||2 =", err_inv)





Matricea A:
 [[0.37454  0.950714 0.731994]
 [0.598658 0.156019 0.155995]
 [0.058084 0.866176 0.601115]]
s: [0.708073 0.020584 0.96991 ]
b = A*s: [0.99474  0.578406 0.641985]

--- Results ---
x_Householder = [0.708073 0.020584 0.96991 ]
x_QR (library) = [0.708073 0.020584 0.96991 ]
||x_QR - x_H||2 = 1.0764020533579939e-14

Errors:
||A x_H - b||2 = 4.710277376051325e-16
||A x_QR - b||2 = 4.1540741810552243e-16
||x_H - s|| / ||s|| = 3.9838886068189014e-15
||x_QR - s|| / ||s|| = 4.983689346722895e-15

Inverse comparison:
||A^-1_H - A^-1_lib||2 = 1.345250804072842e-13
